# NB03 — Reviewer-Proof Ablation, Seeds, and Confidence Intervals

This notebook compares **residual-only vs residual+MMD** across multiple seeds and builds ablation + CI summaries.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install pandas numpy torch scikit-learn scipy

Mounted at /content/drive


In [ ]:
# Paste from the previous strong version if needed.
# This notebook is intentionally included as a ready-to-run seed ablation script.
import os, random, pickle
from dataclasses import dataclass
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import r2_score

@dataclass
class CFG:
    base_dir: str = "/content/drive/MyDrive/ChemDFM/canonical_q1"
    seeds: tuple = (13, 42, 77, 123, 999)
    batch_size: int = 512
    epochs: int = 10
    lr: float = 1e-3
    wd: float = 1e-5
    emb_dim: int = 32
    hidden: int = 256
    dose_hidden: int = 32
    alpha_topk: float = 2.0
    alpha_mmd: float = 0.05
    dropout: float = 0.1
    control_name: str = "control"
cfg = CFG()

ART_DIR = f"{cfg.base_dir}/artifacts"
OUT_DIR = f"{cfg.base_dir}/results_nb03"
os.makedirs(OUT_DIR, exist_ok=True)
X_pca = np.load(f"{ART_DIR}/X_pca.npy")
with open(f"{ART_DIR}/ctrl_means_pca.pkl","rb") as f: ctrl_means_pca = pickle.load(f)
with open(f"{ART_DIR}/drug_encoder.pkl","rb") as f: drug_enc = pickle.load(f)
with open(f"{ART_DIR}/cell_encoder.pkl","rb") as f: cell_enc = pickle.load(f)
obs = pd.read_csv(f"{ART_DIR}/obs_table.csv")
X0_pca = np.stack([ctrl_means_pca[c] for c in obs["cell_type"].astype(str).values]).astype(np.float32)
DELTA_pca = (X_pca - X0_pca).astype(np.float32)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class DS(Dataset):
    def __init__(self, split):
        mask = (obs["_split3"].values == split) & (obs["condition"].astype(str).str.lower().values != cfg.control_name)
        self.idxs = np.where(mask)[0]
    def __len__(self): return len(self.idxs)
    def __getitem__(self, i):
        idx = self.idxs[i]; row = obs.iloc[idx]
        return {"x_true": torch.tensor(X_pca[idx], dtype=torch.float32),
                "x0": torch.tensor(X0_pca[idx], dtype=torch.float32),
                "delta": torch.tensor(DELTA_pca[idx], dtype=torch.float32),
                "drug_idx": torch.tensor(int(row["drug_idx"]), dtype=torch.long),
                "cell_idx": torch.tensor(int(row["cell_idx"]), dtype=torch.long),
                "dose": torch.tensor([np.log1p(max(float(row["dose"]),0.0))], dtype=torch.float32),
                "cell_type": str(row["cell_type"])}

train_ds,test_ds,ood_ds = DS("train"),DS("test"),DS("ood")

class MLP(nn.Module):
    def __init__(self, in_dim, hidden_dims, out_dim, dropout=0.1):
        super().__init__(); layers=[]; prev=in_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev,h), nn.ReLU(), nn.Dropout(dropout)]; prev=h
        layers.append(nn.Linear(prev,out_dim)); self.net=nn.Sequential(*layers)
    def forward(self,x): return self.net(x)

class StructuredDoseEncoder(nn.Module):
    def __init__(self, out_dim=32):
        super().__init__(); self.net=nn.Sequential(nn.Linear(1,16), nn.ReLU(), nn.Linear(16,out_dim))
    def forward(self,dose): return self.net(dose)

class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.drug_emb=nn.Embedding(len(drug_enc.classes_), cfg.emb_dim)
        self.cell_emb=nn.Embedding(len(cell_enc.classes_), cfg.emb_dim)
        self.dose_enc=StructuredDoseEncoder(cfg.dose_hidden)
        self.ctrl_enc=MLP(X_pca.shape[1],[cfg.hidden,cfg.hidden],cfg.hidden,cfg.dropout)
        self.delta_head=MLP(cfg.hidden+cfg.emb_dim+cfg.emb_dim+cfg.dose_hidden,[cfg.hidden,cfg.hidden],X_pca.shape[1],cfg.dropout)
    def forward(self,x0,drug_idx,cell_idx,dose):
        z=torch.cat([self.ctrl_enc(x0),self.drug_emb(drug_idx),self.cell_emb(cell_idx),self.dose_enc(dose)],dim=1)
        delta=self.delta_head(z); return delta, x0+delta

def pairwise_sq_dists(x, y):
    x_norm=(x**2).sum(dim=1, keepdim=True); y_norm=(y**2).sum(dim=1, keepdim=True).T
    return torch.clamp(x_norm+y_norm-2.0*x@y.T, min=0.0)
def rbf_mmd(x,y,gamma=None):
    if x.shape[0]<2 or y.shape[0]<2: return x.new_tensor(0.0)
    dxy=pairwise_sq_dists(x,y); dxx=pairwise_sq_dists(x,x); dyy=pairwise_sq_dists(y,y)
    if gamma is None:
        vals=dxy.flatten(); vals=vals[vals>0]
        gamma = 1.0/(vals.median().item()+1e-6) if vals.numel()>0 else 1.0
    return torch.exp(-gamma*dxx).mean()+torch.exp(-gamma*dyy).mean()-2.0*torch.exp(-gamma*dxy).mean()
def cell_aware_mmd(x_hat,x_true,cell_idx,min_count=8):
    losses=[]
    for cid in torch.unique(cell_idx):
        m=(cell_idx==cid)
        if m.sum().item()>=min_count: losses.append(rbf_mmd(x_hat[m],x_true[m]))
    return torch.stack(losses).mean() if losses else x_hat.new_tensor(0.0)
def topk_mask_from_true(delta_true,k=10):
    idx=torch.topk(delta_true.abs(), k=min(k,delta_true.shape[1]), dim=1).indices
    mask=torch.zeros_like(delta_true); mask.scatter_(1,idx,1.0); return mask
def compute_top50(pred,true,x0):
    vals=[]
    for i in range(true.shape[0]):
        idx=np.argsort(-np.abs(true[i]-x0[i]))[:50]
        vals.append(r2_score(true[i,idx], pred[i,idx]))
    return float(np.mean(vals))
@torch.no_grad()
def eval_loader(model, loader):
    model.eval(); P=[]; T=[]; X0=[]; C=[]
    for b in loader:
        x0=b["x0"].to(DEVICE); x_true=b["x_true"].to(DEVICE); drug_idx=b["drug_idx"].to(DEVICE); cell_idx=b["cell_idx"].to(DEVICE); dose=b["dose"].to(DEVICE)
        _, x_hat = model(x0,drug_idx,cell_idx,dose)
        P.append(x_hat.cpu().numpy()); T.append(x_true.cpu().numpy()); X0.append(x0.cpu().numpy()); C.extend(b["cell_type"])
    P=np.concatenate(P); T=np.concatenate(T); X0=np.concatenate(X0); C=np.array(C)
    return compute_top50(P,T,X0), compute_top50(P[C=="K562"], T[C=="K562"], X0[C=="K562"])

all_rows=[]; all_pc=[]
for seed in cfg.seeds:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    train_loader=DataLoader(train_ds,batch_size=cfg.batch_size,shuffle=True)
    test_loader=DataLoader(test_ds,batch_size=cfg.batch_size,shuffle=False)
    ood_loader=DataLoader(ood_ds,batch_size=cfg.batch_size,shuffle=False)
    print(f"=== Seed {seed}: residual-only ===")
    for variant, alpha_mmd in [("residual_only",0.0),("residual_mmd",cfg.alpha_mmd)]:
        if variant=="residual_mmd": print(f"=== Seed {seed}: residual+MMD ===")
        model=Model().to(DEVICE); opt=torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.wd)
        best_score=-1e9; best=None
        for epoch in range(cfg.epochs):
            model.train()
            for b in train_loader:
                x0=b["x0"].to(DEVICE); x_true=b["x_true"].to(DEVICE); delta_true=b["delta"].to(DEVICE)
                drug_idx=b["drug_idx"].to(DEVICE); cell_idx=b["cell_idx"].to(DEVICE); dose=b["dose"].to(DEVICE)
                opt.zero_grad(); delta_hat,x_hat=model(x0,drug_idx,cell_idx,dose)
                mask=topk_mask_from_true(delta_true,10)
                loss=F.mse_loss(delta_hat,delta_true)+cfg.alpha_topk*((((delta_hat-delta_true)**2)*mask).sum()/(mask.sum()+1e-8))
                if alpha_mmd>0: loss = loss + alpha_mmd*cell_aware_mmd(x_hat,x_true,cell_idx)
                loss.backward(); opt.step()
            test_top50,k562_test=eval_loader(model,test_loader); ood_top50,k562_ood=eval_loader(model,ood_loader)
            score=0.5*test_top50+0.25*ood_top50+0.15*k562_test+0.10*k562_ood
            if score>best_score: best_score=score; best={k:v.cpu().clone() for k,v in model.state_dict().items()}
        model.load_state_dict(best)
        test_top50,k562_test=eval_loader(model,test_loader); ood_top50,k562_ood=eval_loader(model,ood_loader)
        all_rows += [{"seed":seed,"variant":variant,"split":"test","r2_top50":test_top50},{"seed":seed,"variant":variant,"split":"ood","r2_top50":ood_top50}]
        all_pc += [{"seed":seed,"variant":variant,"split":"test","cell_type":"K562","r2_top50":k562_test},{"seed":seed,"variant":variant,"split":"ood","cell_type":"K562","r2_top50":k562_ood}]
overall_df=pd.DataFrame(all_rows); per_cell_df=pd.DataFrame(all_pc)
print(overall_df[["seed","variant","split","r2_top50"]]); print(per_cell_df.head())
ablation_seed_summary=(overall_df.pivot_table(index="split", columns="variant", values="r2_top50", aggfunc="mean").reset_index())
ablation_seed_summary["gain_mmd_over_residual"]=ablation_seed_summary["residual_mmd"]-ablation_seed_summary["residual_only"]
print("ablation_seed_summary"); print(ablation_seed_summary)
k562_paired_ablation=(per_cell_df.pivot_table(index=["seed","split","cell_type"], columns="variant", values="r2_top50").reset_index())
k562_paired_ablation["gain_mmd_over_residual"]=k562_paired_ablation["residual_mmd"]-k562_paired_ablation["residual_only"]
print("\nk562_paired_ablation"); print(k562_paired_ablation)
def bootstrap_ci(arr, n_boot=500, seed=42):
    rng=np.random.default_rng(seed); vals=np.asarray(arr, dtype=float); means=[]
    for _ in range(n_boot):
        idx=rng.choice(len(vals), len(vals), replace=True); means.append(vals[idx].mean())
    lo,hi=np.percentile(means,[2.5,97.5]); return float(vals.mean()), float(lo), float(hi)
ci_rows=[]
for split in ["ood","test"]:
    for variant in ["residual_only","residual_mmd"]:
        vals=overall_df[(overall_df["split"]==split)&(overall_df["variant"]==variant)]["r2_top50"].values
        mean,lo,hi=bootstrap_ci(vals); ci_rows.append({"metric":"r2_top50","split":split,"variant":variant,"mean":mean,"ci95_low":lo,"ci95_high":hi})
        vals2=per_cell_df[(per_cell_df["split"]==split)&(per_cell_df["variant"]==variant)&(per_cell_df["cell_type"]=="K562")]["r2_top50"].values
        mean,lo,hi=bootstrap_ci(vals2); ci_rows.append({"metric":"k562_r2_top50","split":split,"variant":variant,"mean":mean,"ci95_low":lo,"ci95_high":hi})
ci_summary=pd.DataFrame(ci_rows); print("\nci_summary"); print(ci_summary)
overall_df.to_csv(f"{OUT_DIR}/seed_overall_results.csv", index=False)
per_cell_df.to_csv(f"{OUT_DIR}/seed_per_cell_results.csv", index=False)
ablation_seed_summary.to_csv(f"{OUT_DIR}/ablation_seed_summary.csv", index=False)
k562_paired_ablation.to_csv(f"{OUT_DIR}/k562_paired_ablation.csv", index=False)
ci_summary.to_csv(f"{OUT_DIR}/ci_summary.csv", index=False)


=== Seed 13: residual-only ===
=== Seed 13: residual+MMD ===
=== Seed 42: residual-only ===
=== Seed 42: residual+MMD ===
=== Seed 77: residual-only ===
=== Seed 77: residual+MMD ===
=== Seed 123: residual-only ===
=== Seed 123: residual+MMD ===
=== Seed 999: residual-only ===
=== Seed 999: residual+MMD ===
    seed        variant split  r2_top50
0     13  residual_only  test  0.576892
1     13  residual_only   ood  0.556033
2     13   residual_mmd  test  0.576431
3     13   residual_mmd   ood  0.561500
4     42  residual_only  test  0.576841
5     42  residual_only   ood  0.555850
6     42   residual_mmd  test  0.576308
7     42   residual_mmd   ood  0.546587
8     77  residual_only  test  0.577625
9     77  residual_only   ood  0.562105
10    77   residual_mmd  test  0.577370
11    77   residual_mmd   ood  0.557068
12   123  residual_only  test  0.577048
13   123  residual_only   ood  0.555120
14   123   residual_mmd  test  0.575810
15   123   residual_mmd   ood  0.562490
16   999  r